In [20]:
import os
import pandas as pd

old_run_dir = '../results/run_results/clustering/plot_rbd_cluster-2024-02-26_10-25'
tsne_list = os.path.join(old_run_dir, f"rbd_variants_clustering_esm_blstm_a22073_d22073_o22073_iter0_tsne_coordinates.csv")

data_dir = '../data'
tsne_df = pd.read_csv(tsne_list, sep=',', header=0)
meta_data = os.path.join(data_dir, f"spikeprot0528.clean.uniq.noX.RBD.metadata.tsv")
metadata_df = pd.read_csv(meta_data, sep='\t', header=0, low_memory=False)
metadata_df = metadata_df.rename(columns={'Accession ID': 'seq_id'})
merged_df = pd.merge(tsne_df, metadata_df, on='seq_id', how='left')

copy_merged_df = merged_df.copy()
copy_merged_df['AA Substitutions'].unique()

array(['(Spike_H69del,NS8_Q27stop,NSP3_T183I,Spike_G142del,Spike_K1191N,Spike_T716I,NSP6_S106del,N_R203K,NSP2_G339S,Spike_A570D,NSP8_Q24R,NSP13_K460R,Spike_N501Y,NSP3_I1412T,NS8_R52I,NSP16_L163F,Spike_P681H,Spike_Y144del,Spike_F140del,NSP6_G107del,NSP3_A890D,Spike_D1118H,NSP6_F108del,NS8_Y73C,N_G204R,Spike_V70del,Spike_V143del,NSP7_M75I,Spike_P139del,NSP12_P323L,Spike_Y145del,Spike_D614G,N_D3L,NSP16_K160R,Spike_S982A,Spike_L141del,N_S235F)',
       '(Spike_H69del,NS8_Q27stop,NSP3_T183I,Spike_T716I,NSP3_N1080S,NSP6_S106del,N_R203K,Spike_A570D,NSP13_K460R,NSP4_F17L,Spike_N501Y,NSP3_I1412T,NS8_R52I,Spike_P681H,Spike_Y144del,NSP16_L126F,NSP6_G107del,NSP3_A890D,Spike_D1118H,NSP6_F108del,NS8_Y73C,N_G204R,Spike_V503I,Spike_V70del,NSP12_P323L,Spike_D614G,N_D3L,Spike_S982A,N_S235F)',
       '(NSP2_D449E,Spike_H69del,NS8_Q27stop,NS7a_R80T,NSP3_T183I,Spike_T716I,NSP6_S106del,N_R203K,Spike_A570D,Spike_N501Y,NSP3_I1412T,NS8_R52I,Spike_P681H,Spike_Y144del,NSP6_G107del,NSP3_A890D,Spike_D1118H,NSP6_F1

About 118 values are NaN. We need to remove these, however, as they are empty sets. Trying to find the intersection with empty sets will give us an empty set for shared AA Substitutions.

In [21]:
print(len(copy_merged_df[~copy_merged_df['AA Substitutions'].apply(lambda x: isinstance(x, str))]))
copy_merged_df = copy_merged_df.dropna(subset=['AA Substitutions'])
print(len(copy_merged_df[~copy_merged_df['AA Substitutions'].apply(lambda x: isinstance(x, str))]))

118
0


Now, we want to split each entry into a set so that we can find the common and unique AA Substitutions:

In [22]:
copy_merged_df['AA Substitutions Set'] = copy_merged_df['AA Substitutions'].apply(lambda seq: set(seq.strip('()').split(',')))
copy_merged_df['AA Substitutions Set']

0        {Spike_Y145del, NSP6_G107del, NSP3_A890D, Spik...
1        {NSP6_G107del, NSP3_A890D, Spike_D614G, NSP6_F...
2        {NSP6_G107del, NSP3_A890D, Spike_D614G, NSP6_F...
3        {NSP6_G107del, NSP3_A890D, Spike_D614G, NSP6_F...
4        {NSP10_T12I, NSP6_G107del, NSP3_A890D, Spike_D...
                               ...                        
66214    {Spike_Y145del, Spike_A67V, NSP6_G107del, Spik...
66215    {Spike_Y145del, Spike_A67V, NSP6_G107del, Spik...
66216    {NSP6_G107del, Spike_D614G, Spike_G142D, Spike...
66217    {NSP6_G107del, Spike_D614G, Spike_G142D, Spike...
66218    {Spike_L452M, NSP6_G107del, Spike_D614G, Spike...
Name: AA Substitutions Set, Length: 66101, dtype: object

Find the common mutations across all AA Substitutions:

In [23]:
common_aa_substitutions = set.intersection(*copy_merged_df['AA Substitutions Set'])
print(common_aa_substitutions)

alpha_df = copy_merged_df[copy_merged_df['variant'] == 'Alpha']
common_aa_substitutions = set.intersection(*alpha_df['AA Substitutions Set'])
print(common_aa_substitutions)

delta_df = copy_merged_df[copy_merged_df['variant'] == 'Delta']
common_aa_substitutions = set.intersection(*delta_df['AA Substitutions Set'])
print(common_aa_substitutions)

omicron_df = copy_merged_df[copy_merged_df['variant'] == 'Omicron']
common_aa_substitutions = set.intersection(*omicron_df['AA Substitutions Set'])
print(common_aa_substitutions)

set()
set()
set()
set()


Well, still nothing common between all. Let's try to look at the percentages of entries in which each AA Substitution occurs instead. 

In [49]:
from collections import Counter
import pandas as pd

# Flatten the list of all substitutions across all sequences
all_substitutions = [sub for substitutions in copy_merged_df['AA Substitutions Set'] for sub in substitutions]

# Count occurrences of each substitution
substitution_counts = Counter(all_substitutions)

# Calculate the total number of sequences
total_sequences = len(copy_merged_df)

# Calculate the percentage for each substitution
substitution_percentage = {sub: (count / total_sequences) * 100 for sub, count in substitution_counts.items()}

# Convert to a DataFrame for better readability
substitution_percentage_df = pd.DataFrame(list(substitution_percentage.items()), columns=['Substitution', 'Percentage'])

# Sort by percentage
substitution_percentage_df = substitution_percentage_df.sort_values(by='Percentage', ascending=False)

# Reset index
substitution_percentage_df.reset_index(drop=True, inplace=True)
substitution_percentage_df

,Substitution,Percentage
0,Spike_D614G,99.579431
1,NSP12_P323L,98.069621
2,Spike_P681H,65.177531
3,Spike_N501Y,65.101890
4,Spike_T478K,65.000529
...,...,...
41324,NSP2_A247C,0.001513
41325,NSP2_E197Q,0.001513
41326,NSP2_C164V,0.001513
41327,NSP2_Q346E,0.001513


In [74]:
# Now, as a function:

import pandas as pd
from collections import Counter

def calc_substitution_perc(df_column, cutoff_perc=85):
    all_substitutions = [sub for substitutions in df_column for sub in substitutions]
    substitution_counts = Counter(all_substitutions)
    total_sequences = len(df_column)
    substitution_percentage = {sub: (count / total_sequences) * 100 for sub, count in substitution_counts.items()}
    substitution_percentage_df = pd.DataFrame(list(substitution_percentage.items()), columns=['Substitution', 'Percentage'])
    substitution_percentage_df = substitution_percentage_df.sort_values(by='Percentage', ascending=False)
    substitution_percentage_df.reset_index(drop=True, inplace=True)

    # Filter out substitutions below the cutoff percentage
    filtered_df = substitution_percentage_df[substitution_percentage_df['Percentage'] >= cutoff_perc]
    
    return filtered_df

calc_substitution_perc(copy_merged_df['AA Substitutions Set'], cutoff_perc=0)

,Substitution,Percentage
0,Spike_D614G,99.579431
1,NSP12_P323L,98.069621
2,Spike_P681H,65.177531
3,Spike_N501Y,65.101890
4,Spike_T478K,65.000529
...,...,...
41324,NSP2_A247C,0.001513
41325,NSP2_E197Q,0.001513
41326,NSP2_C164V,0.001513
41327,NSP2_Q346E,0.001513


Now compare per variant:

In [81]:
variant_list = copy_merged_df['variant'].unique()
variant_dfs = {}

for variant in variant_list:
    variant_df = copy_merged_df[copy_merged_df['variant'] == variant]
    sub_perc_df = calc_substitution_perc(variant_df['AA Substitutions Set'], cutoff_perc=0)
    variant_dfs[variant] = sub_perc_df

for variant, df in variant_dfs.items():
    print(f"\n{variant}")
    print(df)


Alpha
       Substitution  Percentage
0       Spike_D614G   99.437463
1       NSP12_P323L   99.346731
2        NSP3_T183I   98.697999
3           N_S235F   98.398585
4        NSP3_A890D   98.366828
...             ...         ...
23401    NSP3_A256T    0.004537
23402     NS3_Y184H    0.004537
23403   Spike_Y674C    0.004537
23404  Spike_N1108K    0.004537
23405   Spike_S680T    0.004537

[23406 rows x 2 columns]

Delta
      Substitution  Percentage
0      Spike_D614G   99.718859
1      Spike_T478K   99.165646
2      Spike_P681R   98.902644
3      Spike_L452R   98.553485
4         NS3_S26L   98.453725
...            ...         ...
24954    NSP3_Y87G    0.004535
24955   NSP15_T48C    0.004535
24956   NSP15_L75W    0.004535
24957  NSP15_T112R    0.004535
24958    NSP3_D59A    0.004535

[24959 rows x 2 columns]

Omicron
      Substitution  Percentage
0      Spike_D614G   99.581913
1      Spike_N679K   98.836628
2      Spike_H655Y   98.827539
3      Spike_D796Y   98.563963
4      Spike_P

Let's take a look at the 13 omicron clusters from earlier. Like what we did with pango lineage before, let's try with AA substitutions. 

In [82]:
cluster_aa_data = {}

for i in range(len(omicron_cluster_ranges)):
    # Filter data for the current cluster based on DIM_1 and DIM_2 coordinates
    cluster = omicron_cluster_ranges[i]
    cluster_df = omicron_df[
        (omicron_df['DIM_1'] >= cluster['DIM_1_LEFT']) &
        (omicron_df['DIM_1'] <= cluster['DIM_1_RIGHT']) &
        (omicron_df['DIM_2'] <= cluster['DIM_2_TOP']) &
        (omicron_df['DIM_2'] >= cluster['DIM_2_BOTTOM']) 
    ]
    cluster_aa_substitutions = {}
    sub_perc_df = calc_substitution_perc(cluster_df['AA Substitutions Set'], cutoff_perc=0)
    cluster_aa_data[f"Cluster {i+1}"] = sub_perc_df 
    
    print(f"Omicron Cluster {i+1}:\n{sub_perc_df}\n")

Omicron Cluster 1:
      Substitution  Percentage
0      Spike_S373P  100.000000
1      Spike_H655Y   99.855908
2      Spike_S375F   99.855908
3      Spike_D614G   99.567723
4      Spike_N501Y   99.567723
...            ...         ...
1739    NSP3_A534V    0.144092
1740     NS7a_S36P    0.144092
1741   Spike_P272S    0.144092
1742  Spike_S1170F    0.144092
1743      NSP6_A2T    0.144092

[1744 rows x 2 columns]

Omicron Cluster 2:
     Substitution  Percentage
0     Spike_Y505H  100.000000
1     Spike_D614G  100.000000
2     Spike_N501Y   99.861207
3     Spike_S477N   99.722415
4     Spike_T478K   99.583622
...           ...         ...
3173    NSP1_D33N    0.069396
3174     NS3_L95F    0.069396
3175  NSP12_K281E    0.069396
3176  Spike_L249S    0.069396
3177   NSP14_H26Y    0.069396

[3178 rows x 2 columns]

Omicron Cluster 3:
     Substitution  Percentage
0     Spike_N501Y   99.934598
1     Spike_Y505H   99.934598
2     Spike_D614G   99.869196
3     Spike_S373P   99.803793
4     Spi

Let's try to specify spike AA substitutions, since we used the spike protein sequences for plotting. We'll do all the steps above here, but for spike protein specifically:

In [24]:
# Function to filter for spike protein substitutions
copy_merged_df['Spike Substitutions Set'] = copy_merged_df['AA Substitutions Set'].apply(
    lambda substitutions_set: {sub for sub in substitutions_set if sub.startswith('Spike_')}
)
copy_merged_df['Spike Substitutions Set']

0        {Spike_Y145del, Spike_D614G, Spike_A570D, Spik...
1        {Spike_A570D, Spike_D614G, Spike_H69del, Spike...
2        {Spike_A570D, Spike_D614G, Spike_H69del, Spike...
3        {Spike_A570D, Spike_D614G, Spike_H69del, Spike...
4        {Spike_A570D, Spike_D614G, Spike_H69del, Spike...
                               ...                        
66214    {Spike_Y145del, Spike_A67V, Spike_D614G, Spike...
66215    {Spike_Y145del, Spike_A67V, Spike_D614G, Spike...
66216    {Spike_D614G, Spike_G142D, Spike_A27S, Spike_P...
66217    {Spike_D614G, Spike_Q954H, Spike_G142D, Spike_...
66218    {Spike_L452M, Spike_D614G, Spike_G142D, Spike_...
Name: Spike Substitutions Set, Length: 66101, dtype: object

In [25]:
common_spike_aa_substitutions = set.intersection(*copy_merged_df['Spike Substitutions Set'])
print(common_spike_aa_substitutions )

alpha_df = copy_merged_df[copy_merged_df['variant'] == 'Alpha']
common_spike_aa_substitutions  = set.intersection(*alpha_df['Spike Substitutions Set'])
print(common_spike_aa_substitutions )

delta_df = copy_merged_df[copy_merged_df['variant'] == 'Delta']
common_spike_aa_substitutions  = set.intersection(*delta_df['Spike Substitutions Set'])
print(common_spike_aa_substitutions )

omicron_df = copy_merged_df[copy_merged_df['variant'] == 'Omicron']
common_spike_aa_substitutions = set.intersection(*omicron_df['Spike Substitutions Set'])
print(common_spike_aa_substitutions )

set()
set()
set()
set()


In [75]:
# Whole, spike protein
calc_substitution_perc(copy_merged_df['Spike Substitutions Set'], cutoff_perc=0)

,Substitution,Percentage
0,Spike_D614G,99.579431
1,Spike_P681H,65.177531
2,Spike_N501Y,65.101890
3,Spike_T478K,65.000529
4,Spike_G142D,55.566482
...,...,...
8382,Spike_A694G,0.001513
8383,Spike_E1031D,0.001513
8384,Spike_Q271P,0.001513
8385,Spike_C488W,0.001513


In [80]:
# Variant, spike protein
variant_list = copy_merged_df['variant'].unique()
variant_dfs = {}

for variant in variant_list:
    variant_df = copy_merged_df[copy_merged_df['variant'] == variant]
    sub_perc_df = calc_substitution_perc(variant_df['Spike Substitutions Set'], cutoff_perc=0)
    variant_dfs[variant] = sub_perc_df

for variant, df in variant_dfs.items():
    print(f"\n{variant}")
    print(df)


Alpha
       Substitution  Percentage
0       Spike_D614G   99.437463
1      Spike_D1118H   98.085560
2       Spike_S982A   97.895023
3       Spike_N501Y   97.876877
4       Spike_T716I   97.767999
...             ...         ...
5616  Spike_F238del    0.004537
5617  Spike_G252del    0.004537
5618  Spike_I233del    0.004537
5619  Spike_S221del    0.004537
5620    Spike_S680T    0.004537

[5621 rows x 2 columns]

Delta
       Substitution  Percentage
0       Spike_D614G   99.718859
1       Spike_T478K   99.165646
2       Spike_P681R   98.902644
3       Spike_L452R   98.553485
4        Spike_T19R   98.095497
...             ...         ...
5433  Spike_N764del    0.004535
5434  Spike_N703del    0.004535
5435    Spike_G889L    0.004535
5436    Spike_D737Q    0.004535
5437    Spike_S443C    0.004535

[5438 rows x 2 columns]

Omicron
     Substitution  Percentage
0     Spike_D614G   99.581913
1     Spike_N679K   98.836628
2     Spike_H655Y   98.827539
3     Spike_D796Y   98.563963
4     Spi

In [83]:
# 13 clusters, spike protein
cluster_aa_data = {}

for i in range(len(omicron_cluster_ranges)):
    # Filter data for the current cluster based on DIM_1 and DIM_2 coordinates
    cluster = omicron_cluster_ranges[i]
    cluster_df = omicron_df[
        (omicron_df['DIM_1'] >= cluster['DIM_1_LEFT']) &
        (omicron_df['DIM_1'] <= cluster['DIM_1_RIGHT']) &
        (omicron_df['DIM_2'] <= cluster['DIM_2_TOP']) &
        (omicron_df['DIM_2'] >= cluster['DIM_2_BOTTOM']) 
    ]
    cluster_aa_substitutions = {}
    sub_perc_df = calc_substitution_perc(cluster_df['Spike Substitutions Set'], cutoff_perc=0)
    cluster_aa_data[f"Cluster {i+1}"] = sub_perc_df 
    
    print(f"Omicron Cluster {i+1}:\n{sub_perc_df}\n")

Omicron Cluster 1:
     Substitution  Percentage
0     Spike_S373P  100.000000
1     Spike_H655Y   99.855908
2     Spike_S375F   99.855908
3     Spike_D614G   99.567723
4     Spike_N501Y   99.567723
..            ...         ...
652   Spike_L455S    0.144092
653  Spike_Y1209H    0.144092
654   Spike_K202R    0.144092
655   Spike_R765L    0.144092
656   Spike_T874I    0.144092

[657 rows x 2 columns]

Omicron Cluster 2:
      Substitution  Percentage
0      Spike_D614G  100.000000
1      Spike_Y505H  100.000000
2      Spike_N501Y   99.861207
3      Spike_S477N   99.722415
4      Spike_T478K   99.583622
...            ...         ...
1052   Spike_N657K    0.069396
1053   Spike_N925S    0.069396
1054   Spike_S155N    0.069396
1055  Spike_D1127V    0.069396
1056    Spike_P85Q    0.069396

[1057 rows x 2 columns]

Omicron Cluster 3:
      Substitution  Percentage
0      Spike_Y505H   99.934598
1      Spike_N501Y   99.934598
2      Spike_D614G   99.869196
3      Spike_D405N   99.803793
4    

Let's introduce an 80% cutoff, and save the results. We'll focus on the 1st 3 clusters:

In [91]:
# 13 clusters, spike protein
cluster_aa_data = {}

for i in range(len(omicron_cluster_ranges)):
    # Filter data for the current cluster based on DIM_1 and DIM_2 coordinates
    cluster = omicron_cluster_ranges[i]
    cluster_df = omicron_df[
        (omicron_df['DIM_1'] >= cluster['DIM_1_LEFT']) &
        (omicron_df['DIM_1'] <= cluster['DIM_1_RIGHT']) &
        (omicron_df['DIM_2'] <= cluster['DIM_2_TOP']) &
        (omicron_df['DIM_2'] >= cluster['DIM_2_BOTTOM']) 
    ]
    cluster_aa_substitutions = {}
    sub_perc_df = calc_substitution_perc(cluster_df['Spike Substitutions Set'], cutoff_perc=80)
    cluster_aa_data[f"Cluster {i+1}"] = sub_perc_df 
    
    print(f"Omicron Cluster {i+1}:\n{sub_perc_df}\n")

Omicron Cluster 1:
    Substitution  Percentage
0    Spike_S373P  100.000000
1    Spike_H655Y   99.855908
2    Spike_S375F   99.855908
3    Spike_D614G   99.567723
4    Spike_N501Y   99.567723
5    Spike_N679K   99.567723
6    Spike_Y505H   99.279539
7    Spike_D796Y   99.279539
8    Spike_P681H   99.135447
9    Spike_S477N   99.135447
10   Spike_Q498R   98.847262
11   Spike_T376A   98.703170
12   Spike_N969K   98.703170
13   Spike_Q954H   98.703170
14   Spike_N764K   98.703170
15   Spike_G339D   98.559078
16   Spike_T478K   98.414986
17   Spike_S371F   98.414986
18   Spike_D405N   98.126801
19   Spike_G142D   97.838617
20   Spike_V213G   97.838617
21   Spike_E484A   97.406340
22   Spike_K417N   97.118156
23  Spike_P26del   96.974063
24  Spike_L24del   96.974063
25  Spike_P25del   96.974063
26    Spike_A27S   96.829971
27    Spike_T19I   96.397695
28   Spike_R408S   92.363112
29   Spike_N440K   91.930836
30   Spike_Q493R   91.642651
31   Spike_L452Q   86.167147
32   Spike_S704L   82.85

In [92]:
# Save to csv
all_cluster_dfs = []

for cluster_name, df in cluster_aa_data.items():
    if cluster_name in [f"Cluster {i}" for i in range(4)]:
        print(cluster_name)
        df['Cluster'] = cluster_name
        df = df[['Cluster', 'Substitution', 'Percentage']]
        all_cluster_dfs.append(df)

combined_df = pd.concat(all_cluster_dfs, ignore_index=True)
combined_df.to_csv('small_omicron_clusters_substitutions.csv', index=False)

print(combined_df)

Cluster 1
Cluster 2
Cluster 3
       Cluster Substitution  Percentage
0    Cluster 1  Spike_S373P  100.000000
1    Cluster 1  Spike_H655Y   99.855908
2    Cluster 1  Spike_S375F   99.855908
3    Cluster 1  Spike_D614G   99.567723
4    Cluster 1  Spike_N501Y   99.567723
..         ...          ...         ...
100  Cluster 3  Spike_N460K   96.272073
101  Cluster 3  Spike_Q954H   95.618051
102  Cluster 3  Spike_K444T   95.356442
103  Cluster 3  Spike_R408S   95.029431
104  Cluster 3  Spike_K417N   94.113800

[105 rows x 3 columns]
